**Capstone Project: Supply Chain Monitoring and Optimization Platform**

**Week 4 - Simple ETL in Azure Databricks**

**Capstone Tasks:**
- Upload CSV data into Databricks
- Run a notebook to clean and lter the data
- Save cleaned output as Delta or CSV
- Run basic analysis queries using SQL or PySpark

In [0]:
BASE = "/Volumes/workspace/default/supply_chain" 

ORDERS_PATH    = f"{BASE}/supply_chain_orders.csv"
SUPPLIERS_PATH = f"{BASE}/suppliers.csv"
INVENTORY_PATH = f"{BASE}/inventory.csv"
REPORT_DATE = "2026-10-10"

In [0]:
# 1.Extract
from pyspark.sql import functions as F
 
raw = spark.read.csv(ORDERS_PATH, header=True)             
suppliers = spark.read.csv(SUPPLIERS_PATH, header=True, inferSchema=True)
inventory = spark.read.csv(INVENTORY_PATH, header=True, inferSchema=True)
 
print("Raw order rows:", raw.count())
display(raw)

Raw order rows: 18


order_id,supplier_id,product_id,quantity,order_date,expected_date,delivery_date,status
1,1,1,200,2026-09-01,2026-09-08,2026-09-07,Delivered
2,1,2,300,2026-09-02,2026-09-09,2026-09-12,Delivered
3,2,3,150,2026-09-03,2026-09-13,2026-09-13,Delivered
4,2,4,100,2026-09-05,2026-09-15,2026-09-20,Delivered
5,3,5,1000,2026-09-06,2026-09-10,2026-09-10,Delivered
6,3,6,200,2026-09-08,2026-09-13,2026-09-16,Delivered
7,4,7,200,2026-09-10,2026-09-20,2026-09-19,Delivered
8,4,10,250,2026-09-12,2026-09-22,2026-09-28,Delivered
9,5,8,1000,2026-09-15,2026-09-25,2026-09-25,Delivered
10,5,9,300,2026-09-18,2026-09-28,2026-10-02,Delivered


In [0]:
# 2.Transform (clean + filter)

# CLEAN
clean = (raw
    .dropDuplicates()                                                         
    .dropna(subset=["order_id", "supplier_id", "product_id", "quantity", "expected_date"])
    .withColumn("order_id",    F.col("order_id").cast("int"))
    .withColumn("supplier_id", F.col("supplier_id").cast("int"))
    .withColumn("product_id",  F.col("product_id").cast("int"))
    .withColumn("quantity",    F.col("quantity").cast("int"))
    .withColumn("order_date",    F.to_date("order_date"))
    .withColumn("expected_date", F.to_date("expected_date"))
    .withColumn("delivery_date", F.to_date("delivery_date"))                 
    .withColumn("status", F.initcap(F.trim(F.col("status"))))                
)

In [0]:
# DELAY CALCULATION
# delivered orders: delivery_date; not delivered yet: measured up to REPORT_DATE
end_date = F.coalesce(F.col("delivery_date"), F.to_date(F.lit(REPORT_DATE)))
 
clean = (clean
    .withColumn("delay_days", F.greatest(F.datediff(end_date, F.col("expected_date")), F.lit(0)))
    .withColumn("is_delayed", F.when(F.col("delay_days") > 0, 1).otherwise(0))
    .withColumn("lead_time_days", F.datediff("delivery_date", "order_date"))
)
 
print("Clean order rows:", clean.count())
display(clean.orderBy("order_id"))

Clean order rows: 15


order_id,supplier_id,product_id,quantity,order_date,expected_date,delivery_date,status,delay_days,is_delayed,lead_time_days
1,1,1,200,2026-09-01,2026-09-08,2026-09-07,Delivered,0,0,6
2,1,2,300,2026-09-02,2026-09-09,2026-09-12,Delivered,3,1,10
3,2,3,150,2026-09-03,2026-09-13,2026-09-13,Delivered,0,0,10
4,2,4,100,2026-09-05,2026-09-15,2026-09-20,Delivered,5,1,15
5,3,5,1000,2026-09-06,2026-09-10,2026-09-10,Delivered,0,0,4
6,3,6,200,2026-09-08,2026-09-13,2026-09-16,Delivered,3,1,8
7,4,7,200,2026-09-10,2026-09-20,2026-09-19,Delivered,0,0,9
8,4,10,250,2026-09-12,2026-09-22,2026-09-28,Delivered,6,1,16
9,5,8,1000,2026-09-15,2026-09-25,2026-09-25,Delivered,0,0,10
10,5,9,300,2026-09-18,2026-09-28,2026-10-02,Delivered,4,1,14


In [0]:
# FILTER
delayed = clean.filter(F.col("is_delayed") == 1)                             
low_stock = inventory.filter(F.col("stock_qty") <= F.col("reorder_level"))   
 
print("Delayed shipments:", delayed.count(), "| Products to reorder:", low_stock.count())
display(delayed.select("order_id", "supplier_id", "expected_date", "delivery_date", "delay_days", "status"))

Delayed shipments: 7 | Products to reorder: 4


order_id,supplier_id,expected_date,delivery_date,delay_days,status
8,4,2026-09-22,2026-09-28,6,Delivered
4,2,2026-09-15,2026-09-20,5,Delivered
6,3,2026-09-13,2026-09-16,3,Delivered
2,1,2026-09-09,2026-09-12,3,Delivered
10,5,2026-09-28,2026-10-02,4,Delivered
11,3,2026-09-30,null,10,Delayed
12,2,2026-10-05,null,5,Delayed


In [0]:
# 3.Load - save as Delta tables
(clean.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
      .saveAsTable("supply_chain_orders_clean"))
(delayed.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
        .saveAsTable("supply_chain_delayed_orders"))
(suppliers.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
          .saveAsTable("supply_chain_suppliers"))
(inventory.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
          .saveAsTable("supply_chain_inventory"))
 
display(spark.sql("SHOW TABLES LIKE 'supply_chain*'"))

database,tableName,isTemporary
default,supply_chain_delayed_orders,false
default,supply_chain_inventory,false
default,supply_chain_orders_clean,false
default,supply_chain_suppliers,false


In [0]:
# Save the cleaned data as CSV
OUT = f"{BASE}/output"
clean.coalesce(1).write.mode("overwrite").option("header", True).csv(f"{OUT}/cleaned_orders_csv")
print("CSV saved to", OUT)

CSV saved to /Volumes/workspace/default/supply_chain/output


**4.Analyse**

1.Delayed orders by supplier.

In [0]:
%sql
SELECT s.supplier_name,
COUNT(*) AS delayed_orders,
SUM(o.delay_days) AS total_delay_days,
ROUND(AVG(o.delay_days), 1) AS avg_delay_days
FROM supply_chain_orders_clean o
JOIN supply_chain_suppliers s ON s.supplier_id = o.supplier_id
WHERE o.is_delayed = 1
GROUP BY s.supplier_name
ORDER BY delayed_orders DESC, total_delay_days DESC

supplier_name,delayed_orders,total_delay_days,avg_delay_days
Sagar Packaging,2,13,6.5
Bharat Electronics,2,10,5.0
Deccan Auto Parts,1,6,6.0
Konkan Steel Works,1,4,4.0
Tamil Nadu Textiles,1,3,3.0


2.Delay rate per supplier (all orders)

In [0]:
%sql
SELECT s.supplier_name,
COUNT(*) AS total_orders,
SUM(o.is_delayed) AS delayed_orders,
ROUND(SUM(o.is_delayed) / COUNT(*) * 100, 1) AS delay_rate_pct
FROM supply_chain_orders_clean o
JOIN supply_chain_suppliers s ON s.supplier_id = o.supplier_id
GROUP BY s.supplier_name
ORDER BY delay_rate_pct DESC

supplier_name,total_orders,delayed_orders,delay_rate_pct
Bharat Electronics,3,2,66.7
Sagar Packaging,3,2,66.7
Tamil Nadu Textiles,3,1,33.3
Deccan Auto Parts,3,1,33.3
Konkan Steel Works,3,1,33.3


3.Orders by status

In [0]:
%sql
SELECT status, COUNT(*) AS orders
FROM supply_chain_orders_clean
GROUP BY status
ORDER BY orders DESC

status,orders
Delivered,10
Shipped,2
Delayed,2
Pending,1


4.Products that need reordering

In [0]:
%sql
SELECT product_name, stock_qty, reorder_level, reorder_level - stock_qty AS shortage_qty
FROM supply_chain_inventory
WHERE stock_qty <= reorder_level
ORDER BY shortage_qty DESC

product_name,stock_qty,reorder_level,shortage_qty
Polyester Thread Spool,40,100,60
Lithium Battery Pack,30,60,30
Bubble Wrap Roll,90,100,10
Engine Oil Filter,60,70,10
